In [3]:
from pathlib import Path
import pandas as pd
from datetime import datetime

ROOT = Path(r"C:\Users\keena\Documents\Electoral_Tribes")

EXCLUDE_PARTS = {".git", ".venv", "node_modules", "__pycache__", ".py", ".bat", ".exe", ".fish", ".ps1", ".whl", ".typed", ".pyi", "sklearn-env"}

def classify(path: Path) -> str:
    p = str(path).lower()
    ext = path.suffix.lower()

    if any(part.lower() in EXCLUDE_PARTS for part in path.parts):
        return "exclude"

    if ext in [".gpkg", ".shp", ".geojson", ".kml"]:
        return "boundary_or_geospatial"

    if "lookup" in p or "lu_" in p or "_lu" in p or "mapping" in p:
        return "geography_lookup"

    if ext in [".csv", ".xlsx", ".parquet", ".json"]:
        if "raw" in p:
            return "raw_data"
        if "clean" in p or "interim" in p:
            return "interim_data"
        if "master" in p or "atlas" in p:
            return "master_data"
        if "appendix" in p or "output" in p:
            return "output_table"
        return "data_file_uncategorised"

    if ext == ".ipynb":
        return "notebook"

    if ext in [".py", ".r", ".sql"]:
        return "source_code"

    if ext in [".png", ".jpg", ".jpeg", ".svg", ".pdf"]:
        if "map" in p:
            return "map_or_figure"
        if "report" in p or "draft" in p:
            return "report_or_export"
        return "image_or_document"

    if ext in [".docx", ".txt", ".md"]:
        return "documentation"

    return "other"

records = []

for file in ROOT.rglob("*"):
    if not file.is_file():
        continue

    if any(part in EXCLUDE_PARTS for part in file.parts):
        continue

    stat = file.stat()

    records.append({
        "file_path": str(file),
        "relative_path": str(file.relative_to(ROOT)),
        "file_name": file.name,
        "extension": file.suffix.lower(),
        "folder": str(file.parent.relative_to(ROOT)),
        "size_mb": round(stat.st_size / (1024 * 1024), 3),
        "created_date": datetime.fromtimestamp(stat.st_ctime),
        "modified_date": datetime.fromtimestamp(stat.st_mtime),
        "likely_category": classify(file),
        "project_stage": "",
        "keep_archive_delete": "",
        "canonical_destination": "",
        "sensitive_or_private": "",
        "derived_from": "",
        "used_in_current_model": "",
        "notes": ""
    })

manifest = pd.DataFrame(records)

out = ROOT / "00_ADMIN" / "data_audit"
out.mkdir(parents=True, exist_ok=True)

manifest.to_csv(out / "project_file_manifest_2026-06-01.csv", index=False)
manifest.to_excel(out / "project_file_manifest_2026-06-01.xlsx", index=False)

print(f"Files inventoried: {len(manifest)}")
print(manifest["likely_category"].value_counts())

Files inventoried: 1643
likely_category
map_or_figure              710
data_file_uncategorised    470
output_table               177
boundary_or_geospatial      87
notebook                    47
report_or_export            38
raw_data                    30
documentation               25
interim_data                21
master_data                 18
other                       10
image_or_document            6
geography_lookup             4
Name: count, dtype: int64
